In [44]:
import pandas as pd
from pathlib import Path

# --- find project root (folder that contains /data) ---
def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(10):
        if (p / "data").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Can't find proyect root (not in data parents).")
# C:\dev\projects\heat_mortality_analysis\data\processed\eda_ready\tfe\eda_core_weekly_tfe.parquet
ROOT = find_project_root(Path.cwd())
FP = ROOT / "data" / "processed" / "eda_ready" / "tfe" / "eda_core_weekly_tfe.parquet"
df = pd.read_parquet(FP)
print(df.columns.to_list())


['week_start', 'deaths_week', 'temp_c_mean', 'tmax_c_mean', 'tmin_c_mean', 'humidity_mean', 'pressure_hpa_mean', 'wind_ms_mean', 'prec_sum', 'cap_heat_level_max_week', 'cap_heat_yellow_plus_week', 'cap_dust_level_max_week', 'cap_dust_yellow_plus_week', 'cap_coverage_week', 'calima_any_week', 'calima_any_days_week', 'pm10_mean_week', 'pm25_mean_week', 'sin_doy', 'cos_doy', 'year', 'month', 'weekofyear', 'airq_pm10_ok_week', 'airq_pm25_ok_week', 'airq_pm_pack_ok_week']


In [45]:
cols = ["week_start","deaths_week","tmax_c_mean","pm10_mean_week"]
x = df[cols].copy()
display(x.head())
print(x.dtypes)
print("rows, cols:", x.shape)

,week_start,deaths_week,tmax_c_mean,pm10_mean_week
0,2015-12-28,129.0,23.966667,22.625000
1,2016-01-04,132.0,23.342857,14.558333
2,2016-01-11,162.0,26.042857,16.243506
3,2016-01-18,153.0,24.071429,17.150436
4,2016-01-25,117.0,24.671429,19.382246


week_start        datetime64[ns]
deaths_week              float64
tmax_c_mean              float64
pm10_mean_week           float64
dtype: object
rows, cols: (471, 4)


In [46]:
x["week_start"].min(), x["week_start"].max()

(Timestamp('2015-12-28 00:00:00'), Timestamp('2024-12-30 00:00:00'))

In [47]:
print("x Describe:\n", x["deaths_week"].describe())

x Describe:
 count    471.000000
mean     138.641189
std       21.618765
min       92.000000
25%      123.000000
50%      137.000000
75%      151.000000
max      214.000000
Name: deaths_week, dtype: float64


In [48]:
x["deaths_week"].skew()

np.float64(0.576278616986827)

 <b>Skew positivo y moderado</b>: Nos dice que "hay algunos picos altos"

In [49]:
print("Percentile 99: ", x["deaths_week"].quantile(0.99))

Percentile 99:  195.60000000000002


In [50]:
IQR = x["deaths_week"].quantile(0.75) - x["deaths_week"].quantile(0.25)
print("Dispersion IQR:", IQR)

Dispersion IQR: 28.0


### Temperatura:

In [51]:
temp = x["tmax_c_mean"].copy()
print("Describe:\n", temp.describe())
print("Skew:", temp.skew())
print("Percentile 99:", temp.quantile(0.99))
print("IQR:", temp.quantile(0.75) - temp.quantile(0.25))

Describe:
 count    471.000000
mean      26.019886
std        2.834794
min       20.400000
25%       23.692857
50%       25.757143
75%       28.114286
max       36.942857
Name: tmax_c_mean, dtype: float64
Skew: 0.47034897727692654
Percentile 99: 32.65428571428572
IQR: 4.421428571428571


In [52]:
print(x["week_start"].dtype)
print(x.columns)

datetime64[ns]
Index(['week_start', 'deaths_week', 'tmax_c_mean', 'pm10_mean_week'], dtype='object')


In [53]:
#Estacionalidad
import datetime as dt

x["month"] = x["week_start"].dt.month
est = x.groupby(x["week_start"].dt.month)["tmax_c_mean"].mean()
print(est)


week_start
1     23.072764
2     23.371815
3     23.852068
4     24.544994
5     25.556504
6     26.875985
7     28.883333
8     30.021786
9     28.637343
10    27.996864
11    25.575125
12    23.643274
Name: tmax_c_mean, dtype: float64


In [55]:
m = x.groupby(x["week_start"].dt.month)["tmax_c_mean"].mean()
m.min(), m.max(), (m.max() - m.min())

(np.float64(23.072764227642278),
 np.float64(30.021785714285716),
 np.float64(6.9490214866434385))

“La media mensual de tmax cambia ~7°C entre el mes más frío y el más cálido, lo que es sustancial frente a la variación intra-estación.”

In [56]:
q75 = x["tmax_c_mean"].quantile(0.75)
q25 = x["tmax_c_mean"].quantile(0.25)
iqr = q75 - q25
iqr

np.float64(4.421428571428571)

# Antes de comparar muertes vs calima/PM, siempre mira (y luego ajusta) estacionalidad.

In [57]:
m_deaths = x.groupby(x["week_start"].dt.month)["deaths_week"].mean()
m_deaths.min(), m_deaths.max(), (m_deaths.max() - m_deaths.min())

(np.float64(126.3157894736842),
 np.float64(166.46341463414635),
 np.float64(40.14762516046214))

In [58]:
m_deaths.sort_values().head(3), m_deaths.sort_values(ascending=False).head(3)

(week_start
 9    126.315789
 6    126.864865
 5    127.682927
 Name: deaths_week, dtype: float64,
 week_start
 1     166.463415
 2     156.513514
 12    148.225000
 Name: deaths_week, dtype: float64)

In [60]:
m_pm10 = x.groupby(x["week_start"].dt.month)["pm10_mean_week"].mean()
m_pm10.sort_values().head(3), m_pm10.sort_values(ascending=False).head(3)

(week_start
 6    19.703397
 7    21.562342
 5    22.492245
 Name: pm10_mean_week, dtype: float64,
 week_start
 12    51.158888
 2     42.544642
 1     32.975561
 Name: pm10_mean_week, dtype: float64)